In [8]:
import pandas as pd

master = pd.read_csv("../data/raw/employees_master.csv")
mm = pd.read_csv("../data/raw/employee_monthly_metrics.csv")

locations = pd.read_csv("../data/raw/locations.csv")

mm.info()

<class 'pandas.DataFrame'>
RangeIndex: 17658 entries, 0 to 17657
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   employee_id         17658 non-null  int64  
 1   month               17658 non-null  str    
 2   hours_worked        17658 non-null  float64
 3   overtime_hours      17658 non-null  float64
 4   sales_generated     17602 non-null  float64
 5   customers_served    17658 non-null  int64  
 6   projects_completed  17658 non-null  int64  
 7   absent_days         17658 non-null  int64  
 8   labor_cost          17658 non-null  float64
 9   training_hours      17366 non-null  float64
 10  performance_rating  16909 non-null  float64
dtypes: float64(6), int64(4), str(1)
memory usage: 1.5 MB


In [2]:
master.isna().sum()
# created a copy to not affect the master data. 
employees = master.copy()
# fixed dates str to datetime type
employees["hire_date"] = pd.to_datetime(employees["hire_date"])



nulls counts by columns: 
city                    18
bonus                   79
performance_score       43
manager_id               5
termination_date      1381

- bonus and performance null will affect compensation analysis. also termination date. 
- hire date change from str to datatime. 

HR wants to review employees with missing performance scores.
Starting from the original data:
- show the employees whose performance score is missing
- include:
  - employee
  - department
  - salary
  - performance score
- report how many employees are affected
Then answer:
Would replacing those missing scores with 0 be a reasonable choice here? Why or why not? no because how can we make sure we are not excluding and affecting employees who deserve the bonus based on performance or so? 

In [3]:
nulls = employees["performance_score"].isna().sum()

performance_nulls = (employees[employees["performance_score"].isna()]
 [["employee", "department", "salary", "performance_score"]]
 )
performance_nulls.shape[0]
len(employees["performance_score"])


1500

Business question 2 — harder
Management wants a compensation summary for active employees, but some bonus values are missing.
Create two versions of the analysis:
- Version A: exclude employees whose bonus is missing
- Version B: treat missing bonus as $0
For each version, report:
- number of employees included
- total bonus
- average bonus
- total compensation, where total compensation means salary plus bonus
Then compare the two versions briefly.
The point is not to decide that one is universally correct. The point is to explain how the business assumption changes the result.

In [4]:
null_e = employees["bonus"].isna()
# 1500 employees - 79 nulls
employees[null_e]
# no nulls included version: 
not_nulls = employees[employees["bonus"].notna()]
not_nulls.shape[0] # 1421 employees
not_nulls["bonus"].sum() # total bonus : $5.395.119.99
not_nulls["bonus"].mean().round(3) # average bonus: 3796.707
not_nulls["bonus"].median() # median bonus: 3415.04
not_nulls["total_compensation"] = not_nulls["salary"] + not_nulls["bonus"]
not_nulls["total_compensation"].sum() # total ;$138.250.619.99

#nulls_included version:
employees["bonus"].fillna(0)
employees["bonus"].sum() # total bonus: same as not nulls
employees["bonus"].mean() # same
employees["bonus"].median() # same
employees["total_compensation"] = employees["salary"] + employees["bonus"]
employees["total_compensation"].head(20)

employees.info()


<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   employee_id         1500 non-null   int64         
 1   employee            1500 non-null   str           
 2   department          1500 non-null   str           
 3   city                1482 non-null   str           
 4   salary              1500 non-null   int64         
 5   bonus               1421 non-null   float64       
 6   performance_score   1457 non-null   float64       
 7   projects_completed  1500 non-null   int64         
 8   hire_date           1500 non-null   datetime64[us]
 9   active              1500 non-null   bool          
 10  job_title           1500 non-null   str           
 11  manager_id          1495 non-null   float64       
 12  employment_type     1500 non-null   str           
 13  shift               1500 non-null   str           
 14  loc

Using the employees table, produce a department-level summary that answers all of the following:
- number of employees in each department
- number of active employees in each department
- average salary
- median salary
- average bonus
- average performance score
- total salary expense
- total compensation expense
Then:
- give the output clear column names
- order the departments from highest to lowest total compensation expense
- identify any departments where missing bonus or performance data might affect interpretation
After that, create a second summary broken down by:
department + employment type

- we need to create a table organized by departments and 
then each subsequent column answers each of the points above. 

In [5]:
employees["department"] = employees["department"].str.strip().str.lower()
#num of employees in each department
employees["department"].value_counts()
#num of active employees
len(employees[employees["active"]])
# average salary by department
employees.groupby("department")["salary"].mean().round(2)


 

department
finance       94284.90
hr            80114.21
it           114657.55
marketing     85311.16
sales         84104.21
Name: salary, dtype: float64